[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zhimingkuang/Harvard-AM-115/blob/main/12_earth_age/heat_equation_for_student.ipynb)

# Lord Kelvin Workshop

by Kaylee Vo

In [719]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
import scipy.sparse as sparse
import time
import IPython.display as IP

## Problem 1

### Part A

- Initially we set the radius of the earth ($L$) to be 6370 km, but this made runtime too long. We'll set the radius to be 1000 km, which is one sixth of the radius of the Earth.
- `x = np.linspace(dx, m * dx, num=m, endpoint=True)` because we start at dx, we start one grid point below the surface.

```python

In [720]:
# radius_earth = 6370  # km
# km one sixth of the radius of the Earth
# to make run time quicker
radius_earth = 1000
# Define grid spacing in km
dx = 5
# Define the number of grid points
m = radius_earth // dx
# Define the grid
# We start at dx because we want to the computational domain
# to start one grid point below the surface
x = np.linspace(dx, m * dx, num=m, endpoint=True)
# Integration time
# Let time be in units of million of years
time_e = 100
# Initial condition
# uniform at 2000 degrees Celsius
y0 = np.ones(m) * 2000

In [721]:
m

200

- $x$ is the distance to the center of the Earth, in km.
- $y$ is the temperature at that distance, in degrees Celsius.
- $t$ is in time, in million years.

Now we convert $D$ to be in units of km$^2$/million years.

In [722]:
# thermal diffusivity of rocks is 1.2e-6 m^2/s
D = 1.2e-6 / 1e6  # convert to km^2/second
D = D * (60 * 60 * 24 * 365.25)  # convert to km^2/year
D = D * 1e6  # convert to km^2/million years

In [723]:
D

37.86912

In [724]:
# Define the Laplacian operator
e1 = np.ones((1, m))  # build a vector of ones
diags = np.concatenate((e1, -2.0 * e1, e1))  # diagonal entries
offsets = np.array([-1, 0, 1])  # which diagonals

A = sparse.dia_matrix((diags, offsets), shape=(m, m))  # diagonal matrix
A = A.tolil()
# A[0, :] = 0
A[0, 0] = -2
A[-1, -1] = -1
A = A.tocsr()

A = A.tocsr()  # convert to CSR format for efficiency

In [725]:
A.shape

(200, 200)

In [726]:
A.toarray()[:5, :5]  # print the first 5 rows and columns

array([[-2.,  1.,  0.,  0.,  0.],
       [ 1., -2.,  1.,  0.,  0.],
       [ 0.,  1., -2.,  1.,  0.],
       [ 0.,  0.,  1., -2.,  1.],
       [ 0.,  0.,  0.,  1., -2.]])

In [727]:
A.toarray()[-5:, -5:]  # print the first 5 rows and columns

array([[-2.,  1.,  0.,  0.,  0.],
       [ 1., -2.,  1.,  0.,  0.],
       [ 0.,  1., -2.,  1.,  0.],
       [ 0.,  0.,  1., -2.,  1.],
       [ 0.,  0.,  0.,  1., -1.]])

In [728]:
# Source term
b = np.zeros(m)
b[0] = 0.0  # Surface


# Use ode solvers for this; method of lines
def F(t, y, D, A, b, dx):
    """RHS of the heat equation

    Args:
        t (float): 1-D independent variable (time)
        y (numpy.ndarray): N-D vector-valued function (state)
        D (float): diffusivity
        A (numpy.ndarray): Laplacian operator
        b (numpy.ndarray): source term
        dx (float): grid spacing

    Returns:
        numpy.ndarray: differential equation

    """
    return D * (sparse.csr_matrix.dot(A, y) + b) / (dx**2)


sol = solve_ivp(
    F,
    [0, time_e],
    y0,
    args=(D, A, b, dx),
    max_step=1.0,
    dense_output=True,
    method="Radau",  # Runge-Kutta method
    t_eval=np.linspace(0, time_e, 100),  # evaluate at these
    options={"atol": 1e-6, "rtol": 1e-6},  # set tolerances for the solver
)
t = sol.t
y = sol.y

In [729]:
# Customize for matplotlib
# If interested in the matplotlib object hierarchy, check: https://realpython.com/python-matplotlib-guide/
plt.rcParams["axes.linewidth"] = 1
plt.rcParams["xtick.bottom"] = True
plt.rcParams["ytick.left"] = True
plt.rcParams["xtick.direction"] = "in"
plt.rcParams["ytick.direction"] = "in"
plt.rcParams["mathtext.default"] = "regular"
# Change font size: http://www.futurile.net/2016/02/27/matplotlib-beautiful-plots-with-style/
plt.rcParams["font.size"] = 12
plt.rcParams["axes.labelsize"] = 14
plt.rcParams["axes.labelweight"] = "bold"
plt.rcParams["xtick.labelsize"] = 12
plt.rcParams["ytick.labelsize"] = 12
plt.rcParams["legend.fontsize"] = 14
plt.rcParams["figure.titlesize"] = 20

In [730]:
# # Alternative way to animate the solution using FuncAnimation
# fig = plt.figure(figsize=(9, 7))
# ax = fig.add_subplot(1, 1, 1)

# for i, time in enumerate(t):
#     ax.cla()
#     ax.plot(x, y[:, i], "r-o")
#     ax.set_xlim(0, radius_earth)
#     ax.set_ylim(0, 2200)
#     ax.set_xlabel(r"$x$")
#     ax.set_ylabel(r"$y$")
#     ax.set_title(r"$t = %.3f$ million of years" % time)

#     IP.display(fig)
#     IP.clear_output(wait=True)
#     plt.pause(0.01)

In [731]:
# dimensions are
# (distance to the center of the Earth in km, and time)
y.shape

(200, 100)

In [732]:
threshold = 25  # °C/km

for t, y_t in enumerate(sol.t):

    gradient = (y[:, t][0] - 0) / dx

    if gradient <= threshold:
        print(round(y_t, 2), "million years")
        break

54.55 million years


In [733]:
# reduced the radius of the Earth to 1/6

# Case 0:
# dx = 50 (50 km)
# 42.42 million years

# Case 1:
# dx = 10 (10 km)
# 53.54 million years

# Case 2:
# dx = 5 (5 km)
# 54.55 million years

# Case 3:
# dx = 1 (1 km)
# 54.55 million years

In [734]:
def analytical_solution(x, time_e, D):

    y0 = 2000
    ys = 0
    # This is the change of temperature for a single time step
    soln = (y0 - ys) / np.sqrt(np.pi * D * time_e) * np.exp(-(x**2) / (4 * D * time_e))

    return soln


for t in np.arange(1, 100, 0.1):
    temp_gradient = analytical_solution(x, t, D)[0]
    if temp_gradient < threshold:
        t = round(t, 2)
        temp_gradient = round(temp_gradient, 2)
        print(f"Time step {t}: Temperature change = {temp_gradient} °C/km")
        print(f"Age of Earth determined by analytical solution: {t} million years")
        break

Time step 53.5: Temperature change = 24.99 °C/km
Age of Earth determined by analytical solution: 53.5 million years


Is 1d sphere reasonable?
- only reasonable if the temperature is uniform across the earth.
- only reasonable if the earth is a sphere, which I guess is reasonable (not a reason)
- If most of the gradient change is near the surface, then it is reasonable to model the earth as a 1D sphere.